In [1]:
import pandas as pd
import os
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

In [2]:
folders = [
    "models",
    "outputs",
    "outputs/tables"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Folders created successfully.")

Folders created successfully.


In [3]:
df = pd.read_csv("data/medical_text_data.csv")

df.head()

,text,category,clean_text,text_length,word_count
0,I have been experiencing chest discomfort and ...,Symptoms enquiry,i have been experiencing chest discomfort and ...,66,10
1,My child has a high fever and a sore throat.,Symptoms enquiry,my child has a high fever and a sore throat,44,10
2,I feel dizzy and weak most mornings.,Symptoms enquiry,i feel dizzy and weak most mornings,36,7
3,I have a persistent headache and blurred vision.,Symptoms enquiry,i have a persistent headache and blurred vision,48,8
4,I am coughing and my body feels very tired.,Symptoms enquiry,i am coughing and my body feels very tired,43,9


In [4]:
print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns)

print("\nCategory distribution:")
print(df["category"].value_counts())

Dataset shape: (60, 5)

Columns:
Index(['text', 'category', 'clean_text', 'text_length', 'word_count'], dtype='object')

Category distribution:
category
Symptoms enquiry        10
Medication enquiry      10
Chronic care enquiry    10
Medical aid benefits    10
Claims enquiry          10
Emergency guidance      10
Name: count, dtype: int64


In [5]:
X = df["clean_text"]
y = df["category"]

print("Number of text samples:", len(X))
print("Number of categories:", y.nunique())

Number of text samples: 60
Number of categories: 6


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 48
Testing samples: 12


In [7]:
models = {
    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]),

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", MultinomialNB())
    ]),

    "Support Vector Machine": Pipeline([
        ("tfidf", TfidfVectorizer(
            stop_words="english",
            ngram_range=(1, 2),
            max_features=3000
        )),
        ("classifier", LinearSVC(
            random_state=42
        ))
    ])
}

In [8]:
results = []
trained_models = {}

for model_name, model_pipeline in models.items():
    print(f"\nTraining model: {model_name}")
    
    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    
    cv_scores = cross_val_score(
        model_pipeline,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )
    
    results.append({
        "model": model_name,
        "test_accuracy": accuracy,
        "mean_cv_accuracy": cv_scores.mean(),
        "std_cv_accuracy": cv_scores.std()
    })
    
    trained_models[model_name] = model_pipeline
    
    print(f"Test Accuracy: {accuracy:.4f}")
    print(f"Mean CV Accuracy: {cv_scores.mean():.4f}")
    print(f"CV Std Dev: {cv_scores.std():.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))


Training model: Logistic Regression
Test Accuracy: 0.5000
Mean CV Accuracy: 0.6867
CV Std Dev: 0.0959

Classification Report:
                      precision    recall  f1-score   support

Chronic care enquiry       0.00      0.00      0.00         2
      Claims enquiry       0.50      1.00      0.67         2
  Emergency guidance       0.33      0.50      0.40         2
Medical aid benefits       1.00      0.50      0.67         2
  Medication enquiry       0.00      0.00      0.00         2
    Symptoms enquiry       0.50      1.00      0.67         2

            accuracy                           0.50        12
           macro avg       0.39      0.50      0.40        12
        weighted avg       0.39      0.50      0.40        12


Training model: Naive Bayes
Test Accuracy: 0.3333
Mean CV Accuracy: 0.6889
CV Std Dev: 0.1456

Classification Report:
                      precision    recall  f1-score   support

Chronic care enquiry       0.00      0.00      0.00         2
      

In [9]:
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="test_accuracy",
    ascending=False
).reset_index(drop=True)

results_df

,model,test_accuracy,mean_cv_accuracy,std_cv_accuracy
0,Support Vector Machine,0.666667,0.773333,0.094699
1,Logistic Regression,0.500000,0.686667,0.095942
2,Naive Bayes,0.333333,0.688889,0.145551


In [10]:
results_df.to_csv("outputs/tables/model_comparison_results.csv", index=False)

print("Model comparison table saved successfully.")

Model comparison table saved successfully.


In [11]:
best_model_name = results_df.loc[0, "model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)

Best model: Support Vector Machine


In [12]:
model_path = "models/best_medical_text_classifier.pkl"

joblib.dump(best_model, model_path)

print(f"Best model saved to: {model_path}")

Best model saved to: models/best_medical_text_classifier.pkl


In [13]:
new_texts = [
    "I need to know if my medical aid covers dental treatment.",
    "My chronic medication authorisation has expired.",
    "I submitted a claim but it has not been paid.",
    "I have chest pain and I am struggling to breathe.",
    "Can I take this antibiotic with food?",
    "Where can I find urgent emergency medical help?"
]

predictions = best_model.predict(new_texts)

prediction_df = pd.DataFrame({
    "medical_text": new_texts,
    "predicted_category": predictions
})

prediction_df

,medical_text,predicted_category
0,I need to know if my medical aid covers dental...,Medical aid benefits
1,My chronic medication authorisation has expired.,Chronic care enquiry
2,I submitted a claim but it has not been paid.,Claims enquiry
3,I have chest pain and I am struggling to breathe.,Emergency guidance
4,Can I take this antibiotic with food?,Medication enquiry
5,Where can I find urgent emergency medical help?,Emergency guidance


In [14]:
prediction_df.to_csv("outputs/tables/example_predictions.csv", index=False)

print("Example predictions saved successfully.")

Example predictions saved successfully.


In [15]:
files_to_check = [
    "outputs/tables/model_comparison_results.csv",
    "outputs/tables/example_predictions.csv",
    "models/best_medical_text_classifier.pkl"
]

for file in files_to_check:
    print(file, "exists:", os.path.exists(file))

outputs/tables/model_comparison_results.csv exists: True
outputs/tables/example_predictions.csv exists: True
models/best_medical_text_classifier.pkl exists: True
